In [8]:
import torch
import torch.nn as nn
import math
import random

device = "cuda"

batch_size = 64
block_size = 256 # T
n_embed = 384
p = 0.2
num_heads = 6
num_layers = 6

lr = 3e-4
max_iters = 5000

f = open("/kaggle/input/datasets/aidenjeong/input-txt/input.txt", "r").read()

chars = sorted(list(set(f)))
vocab_size = len(chars)

chartonum = {c : i for i,c in enumerate(chars)}
numtochar = {i : c for i,c in enumerate(chars)}

encode = lambda s: [chartonum[t] for t in s]
decode = lambda x: [numtochar[y] for y in x]

data = torch.tensor(encode(f))
train = data[:math.floor(len(data)*0.9)]
val = data[math.floor(len(data)*0.9):]

def get_batch(where):
    x = torch.empty(batch_size,block_size, dtype = torch.long)
    y = torch.empty(batch_size,block_size, dtype = torch.long)
    for k in range(batch_size):
        i = random.randint(0,len(where)-block_size-2) # 2?
        x[k]=where[i:i+block_size]
        y[k]=where[i+1:i+block_size+1]
    return x.to(device), y.to(device)

class Head(nn.Module):
    def __init__(self, head_size): # d_k 
        super().__init__()
        self.key = nn.Linear(n_embed, head_size)
        self.query = nn.Linear(n_embed, head_size)
        self.value = nn.Linear(n_embed, head_size)
        self.dropout = nn.Dropout(p)

    def forward(self, x): # B, T, n_embed
        k = self.key(x) # B, T, hs
        q = self.query(x)
        v = self.value(x)

        att = q @ torch.transpose(k, -1, -2)/math.sqrt(k.size(-1)) # B, T, T

        mask = -float('inf')*torch.ones(k.size(-2),k.size(-2), device = device)
        mask = torch.tril(mask,diagonal=-1).transpose(-1,-2)

        att = torch.softmax(att + mask, dim = -1)
        att = self.dropout(att)

        return att @ v # B, T, hs
    
class MultiHead(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.w = nn.Linear(num_heads*head_size, n_embed)
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.dropout = nn.Dropout(p)
    
    def forward(self, x):
        after_heads = torch.concat([H(x) for H in self.heads], dim = -1) # B, T, hx*nh
        
        resize = self.w(after_heads) # B, T, n_embed
        resize = self.dropout(resize)

        return resize
    
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.ff = nn.Sequential(
            nn.Linear(n_embed, 4*n_embed),
            nn.ReLU(),
            nn.Linear(4*n_embed, n_embed),
            nn.Dropout(p)
        )
    def forward(self, x):
        return self.ff(x)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.mha = MultiHead(num_heads=num_heads, head_size = n_embed // num_heads)
        self.ff = FeedForward()
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.mha(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.ModuleList([Block() for _ in range(num_layers)])
        self.ln = nn.LayerNorm(n_embed)
        self.resize = nn.Linear(n_embed, vocab_size)
        
        self.positional_embedding = nn.Embedding(num_embeddings=block_size, embedding_dim=n_embed)
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=n_embed)

    def inference(self, x): # x has size B, T or just T
        pos = torch.arange(block_size, device = device)
        x = self.embedding(x) + self.positional_embedding(pos)

        for B in self.blocks:
            x = B.forward(x)
        x = self.ln(x)
        x = self.resize(x)
        return x

    def generate(self, input, count): # input is K tensor where K >> block_size
        final = ""
        for _ in range(count):
            probs = torch.softmax(self.inference(input[-block_size:]), dim = -1)
            next = torch.argmax(probs[block_size-1])
            next = torch.tensor([next]).to(device)
            input = torch.cat((input, next))
            final += numtochar[next.item()]
        return final

model = Model().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr = lr)
loss_fn = nn.CrossEntropyLoss()

for iter in range(max_iters):
    x,y = get_batch(train)
    z = model.inference(x)

    y = torch.reshape(y, (-1,))
    z = torch.reshape(z, (-1,vocab_size))
    loss = loss_fn(z,y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if iter % 100 == 0:
        print(iter, loss.item())

0 4.328773498535156
100 2.4863526821136475
200 2.4140663146972656
300 2.2587718963623047
400 2.1088342666625977
500 1.9615862369537354
600 1.854706048965454
700 1.7961081266403198
800 1.7424249649047852
900 1.6348482370376587
1000 1.6403231620788574
1100 1.5654953718185425
1200 1.5672147274017334
1300 1.5370209217071533
1400 1.4895317554473877
1500 1.4799425601959229
1600 1.4539846181869507
1700 1.4212702512741089
1800 1.4281357526779175
1900 1.4229705333709717
2000 1.3857204914093018
2100 1.3893589973449707
2200 1.381541132926941
2300 1.3629627227783203
2400 1.3577958345413208
2500 1.315071940422058
2600 1.3009421825408936
2700 1.2955167293548584
2800 1.309175968170166
2900 1.2672265768051147
3000 1.277609944343567
3100 1.301100730895996
3200 1.2467882633209229
3300 1.272963523864746
3400 1.2308595180511475
3500 1.2580194473266602
3600 1.2479606866836548
3700 1.239919662475586
3800 1.2342864274978638
3900 1.217475414276123
4000 1.231399416923523
4100 1.2129995822906494
4200 1.21806693

In [12]:
print(model.generate(val.to(device),1000))


MENENIUS:
Not thy country's name?

COMINIUS:
No, no, no, no.

CORIOLANUS:
The gods contend the courtesy.

CORIOLANUS:
The gods he have seen the gods place.

MENENIUS:
The word will not be the world to seek the world
be the world of the people.

CORIOLANUS:
The gods but that he was a man of straight
To see the people of the way and the world to have
with the prince of the second the world
That he hath set the seal of the world well.

CORIOLANUS:
The gods have stroken the state of the world,
The second that he stands the state of the state,
That he hath sended to the senate of the sea
To see how the secret he was to be the sense,
And the sentence of the seat of the senate,
That the senate of the senate of the sea
That the streaten of the world that have strengthen
The sentence of the senate of the sea,
The second the senate of the season of the world,
The sentence of the second and the seals
Of the senate that the state of the sense,
The strew'st the stretchest that the seas,
That the s